# Anomaly detection on the Kaggle UR3e pick-and-place IMU data

**Dataset:** *Industrial Robotic Arm – Pick and Place* (Kaggle, `hkayan`) — a 9-axis IMU stream
(`AccX, AccY, AccZ, GyroX, GyroY, GyroZ, MagX, MagY, MagZ`) recorded from a UR3e arm.
The file has **no label column**, so this notebook uses **unsupervised** learning:
learn what *normal* motion looks like, then flag windows that are far from it.

**What the notebook does**
1. Load the CSV (your 1000-row file) and look at it
2. Cut the signal into sliding windows and compute features (mean and std of every channel)
3. Build two detectors: a **k-NN novelty detector** (same method as the simulator's ML panel) and an **Isolation Forest** for comparison
4. Score every window with **blocked cross-validation** (each stretch of the recording is judged by a model that never saw it)
5. Measure performance **without labels**: precision / recall / F1 / ROC-AUC on **injected faults** whose positions we know, plus how often the original recording is flagged
6. Explain which sensors drive the alarms, train the final model on the whole recording and save it

**How to run:** put your CSV next to this notebook as `imu_pick_place.csv` (or change `CSV_PATH`).
In Google Colab, just run the notebook and upload the file when asked.

In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# ---------------- configuration ----------------
CSV_PATH     = "imu_pick_place.csv"   # your uploaded 1000-row CSV
WINDOW       = 25                     # samples per window
STRIDE       = 5                      # step between windows
BLOCK        = 10                     # cross-validation block = 10 windows (~50 samples)
K_NEIGHBORS  = 5                      # k of the k-NN novelty score
THRESH_MULT  = 1.5                    # alarm threshold = THRESH_MULT x (99th percentile of normal scores)
RANDOM_STATE = 42
CHANNELS = ["AccX","AccY","AccZ","GyroX","GyroY","GyroZ","MagX","MagY","MagZ"]
rng = np.random.default_rng(RANDOM_STATE)

## 1 · Load the data

In [ ]:
if not os.path.exists(CSV_PATH):
    try:                                   # Google Colab: ask for the file
        from google.colab import files
        uploaded = files.upload()
        CSV_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Put your CSV next to this notebook as '{CSV_PATH}' (or edit CSV_PATH).")

df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]
missing = [c for c in CHANNELS if c not in df.columns]
assert not missing, f"CSV is missing columns: {missing}"
df = df[CHANNELS].apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)
X = df.values.astype(float)
print(f"{len(df)} valid rows, {len(CHANNELS)} channels")
df.head()

In [ ]:
df.describe().T

### Look at the signals
The arm sits still for a while (flat lines, only sensor noise), then moves: the gyroscope jumps to tens of °/s while it picks and places.

In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
for c in CHANNELS[:3]:  ax[0].plot(df[c], label=c, lw=.8)
for c in CHANNELS[3:6]: ax[1].plot(df[c], label=c, lw=.8)
for c in CHANNELS[6:]:  ax[2].plot(df[c], label=c, lw=.8)
ax[3].plot(np.linalg.norm(X[:, 3:6], axis=1), color="C1", lw=.9)
for a, t in zip(ax, ["Accelerometer (m/s²)", "Gyroscope (°/s)", "Magnetometer (µT)", "|Gyro| = overall rotation speed"]):
    a.set_title(t, loc="left", fontsize=10); a.grid(alpha=.3)
    if a is not ax[3]: a.legend(loc="upper right", ncol=3, fontsize=8)
ax[3].set_xlabel("sample"); plt.tight_layout(); plt.show()

## 2 · Windows and features
A single sample says little; a short **window** shows the behaviour. For each window of 25 samples we compute the **mean** and **standard deviation**
of every channel → 18 features per window. The mean captures *where* the arm is, the std captures *how violently* it moves or vibrates.

In [ ]:
def make_windows(X, win=WINDOW, stride=STRIDE):
    starts = np.arange(0, len(X) - win + 1, stride)
    F = np.empty((len(starts), 2 * X.shape[1]))
    for i, s in enumerate(starts):
        w = X[s:s + win]
        F[i, 0::2] = w.mean(0)
        F[i, 1::2] = w.std(0)
    return starts, F

FEATURES = [f"{c}_{k}" for c in CHANNELS for k in ("mean", "std")]
starts, F = make_windows(X)
print(f"{len(X)} rows -> {len(F)} windows x {F.shape[1]} features")
pd.DataFrame(F[:5], columns=FEATURES).round(3)

## 3 · Model A — k-NN novelty detector
**Idea:** keep the normal training windows as a *reference library*. A new window is **normal** if it has close look-alikes in the library
(its distance to the 5th-nearest reference is small) and **anomalous** if nothing in the library resembles it.

*Calibration:* to know what a "normal" distance is, we score the training windows themselves — but neighbouring windows overlap and are almost identical,
so we ignore the overlapping neighbours (otherwise every distance would look tiny). Threshold = `THRESH_MULT × 99th-percentile` of those normal scores.

In [ ]:
class KNNNovelty:
    def __init__(self, k=K_NEIGHBORS, mult=THRESH_MULT, win=WINDOW, stride=STRIDE):
        self.k, self.mult = k, mult
        self.exclude = int(np.ceil(win / stride)) + 1          # windows closer than this overlap in time
    def fit(self, F):
        self.scaler = StandardScaler().fit(F)
        self.Z = self.scaler.transform(F)
        n_nb = min(len(self.Z), self.k + 2 * self.exclude + 2)
        self.nn = NearestNeighbors(n_neighbors=n_nb).fit(self.Z)
        d, idx = self.nn.kneighbors(self.Z)
        sc = []
        for i in range(len(self.Z)):
            dd = d[i][np.abs(idx[i] - i) > self.exclude]          # drop self + overlapping neighbours
            sc.append(dd[self.k - 1] if len(dd) >= self.k else dd[-1])
        self.train_scores = np.array(sc) / np.sqrt(self.Z.shape[1])
        self.q99 = np.quantile(self.train_scores, 0.99)
        return self
    @property
    def threshold(self): return self.q99 * self.mult
    def score(self, F):
        d, _ = self.nn.kneighbors(self.scaler.transform(F), n_neighbors=self.k)
        return d[:, -1] / np.sqrt(self.Z.shape[1])
    def predict(self, F): return self.score(F) > self.threshold
    def explain(self, F):
        """mean |difference| to the nearest normal window, per channel (mean+std features averaged)"""
        Z = self.scaler.transform(F); _, idx = self.nn.kneighbors(Z, n_neighbors=1)
        diff = np.abs(Z - self.Z[idx[:, 0]])
        return diff.reshape(len(F), len(CHANNELS), 2).mean(2)

knn = KNNNovelty().fit(F)
print(f"normal-score 99th percentile = {knn.q99:.3f}  ->  alarm threshold = {knn.threshold:.3f}")
plt.figure(figsize=(8, 3)); plt.hist(knn.train_scores, bins=40, color="C0"); plt.axvline(knn.threshold, color="r", ls="--", label="threshold")
plt.title("Novelty scores of the training windows (overlapping neighbours ignored)"); plt.xlabel("score"); plt.legend(); plt.show()

## 4 · Model B — Isolation Forest (for comparison)
A standard unsupervised detector: it isolates points with random splits; points that are isolated quickly are anomalies.
`contamination=0.01` means we expect about 1 % of the training windows to be odd.

In [ ]:
class IForest:
    def __init__(self): pass
    def fit(self, F):
        self.scaler = StandardScaler().fit(F)
        self.m = IsolationForest(n_estimators=300, contamination=0.01, random_state=RANDOM_STATE).fit(self.scaler.transform(F))
        return self
    def score(self, F):   return -self.m.score_samples(self.scaler.transform(F))     # higher = more anomalous
    def predict(self, F): return self.m.predict(self.scaler.transform(F)) == -1

## 5 · Inject known faults (so we can measure precision / recall)
We add faults whose positions we *know* to a copy of the recording: **vibration bursts**, **short violent spikes** and **frozen-sensor dropouts**.

In [ ]:
def inject_faults(X, rng, n_events=5, spacing=120):
    """returns a corrupted copy of X and a per-sample boolean label (True = injected fault)"""
    X2, lab = X.copy(), np.zeros(len(X), bool)
    slots = np.arange(20, len(X) - spacing, spacing); rng.shuffle(slots)
    kinds = ["burst", "spike", "dropout"]
    for n, s0 in enumerate(slots[:n_events]):
        kind = kinds[n % 3]
        if kind == "burst":                       # vibration: heavy noise on accel + gyro
            L = 30
            X2[s0:s0+L, :3]  += rng.normal(0, 1.2, (L, 3))
            X2[s0:s0+L, 3:6] += rng.normal(0, 15,  (L, 3))
        elif kind == "spike":                     # short violent jolt
            L = 5
            X2[s0:s0+L, :3]  += rng.choice([-1, 1]) * 4
            X2[s0:s0+L, 3:6] += rng.choice([-1, 1]) * 40
        else:                                     # sensor freeze
            L = 20
            X2[s0:s0+L] = X2[s0]
        lab[s0:s0+L] = True
    return X2, lab

X_bad, lab = inject_faults(X, rng)
_, F_bad = make_windows(X_bad)
y_true = np.array([lab[s:s+WINDOW].any() for s in starts])
print(f"{lab.sum()} faulty samples -> {y_true.sum()} faulty windows out of {len(y_true)}")

## 6 · Blocked cross-validation — judge each stretch with a model that has not seen it
Cut the windows into blocks of 10. For every block: train on the other windows (leaving a **gap** around the block so overlapping windows cannot leak),
then score that block twice — once on the **original** data and once on the **corrupted** data. Collecting all blocks gives a fair score for every window in the recording.

In [ ]:
GAP = int(np.ceil(WINDOW / STRIDE)) + 1
n = len(F)
S = {m: {"clean": np.zeros(n), "bad": np.zeros(n), "pc": np.zeros(n, bool), "pb": np.zeros(n, bool)} for m in ("knn", "iso")}
for b0 in range(0, n, BLOCK):
    test = np.arange(b0, min(n, b0 + BLOCK))
    train = np.array([i for i in range(n) if i < b0 - GAP or i >= b0 + BLOCK + GAP])
    if len(train) < 30: continue
    for name, mdl in (("knn", KNNNovelty().fit(F[train])), ("iso", IForest().fit(F[train]))):
        S[name]["clean"][test], S[name]["bad"][test]  = mdl.score(F[test]), mdl.score(F_bad[test])
        S[name]["pc"][test],    S[name]["pb"][test]   = mdl.predict(F[test]), mdl.predict(F_bad[test])
print("done:", n, "windows scored with", int(np.ceil(n / BLOCK)), "blocks")

In [ ]:
def report(name, key):
    sc, pr = S[key]["bad"], S[key]["pb"]
    return {"model": name,
            "precision": precision_score(y_true, pr, zero_division=0),
            "recall":    recall_score(y_true, pr, zero_division=0),
            "F1":        f1_score(y_true, pr, zero_division=0),
            "ROC-AUC":   roc_auc_score(y_true, sc),
            "flagged in ORIGINAL recording": S[key]["pc"].mean()}
results = pd.DataFrame([report("k-NN novelty", "knn"), report("Isolation Forest", "iso")]).set_index("model").round(3)
results

**Metrics in plain words**
- **Precision** – of the windows we flagged, how many were really faulty (low → many false alarms)
- **Recall** – of the really faulty windows, how many we caught (low → missed faults)
- **F1** – one number balancing precision and recall
- **ROC-AUC** – threshold-free quality of the *score* (1.0 = perfect ranking, 0.5 = random guessing)
- **flagged in ORIGINAL recording** – share of the untouched recording that was flagged. We assume the recording is mostly normal, so this should be small;
  if a stretch is flagged, look at it — it is either a false alarm or a *genuine* anomaly in the data.

In [ ]:
def shade(ax, starts, flags, win=WINDOW, color="red", alpha=.25):
    for s0, f in zip(starts, flags):
        if f: ax.axvspan(s0, s0 + win, color=color, alpha=alpha, lw=0)

fig, ax = plt.subplots(3, 1, figsize=(11, 8.5), sharex=True)
ax[0].plot(np.linalg.norm(X_bad[:, 3:6], axis=1), color="C1", lw=.9); ax[0].set_title("|Gyro| of the corrupted recording (green = injected fault)", loc="left")
ax[0].fill_between(np.arange(len(lab)), 0, lab * ax[0].get_ylim()[1], color="green", alpha=.25, step="mid")
ctr = starts + WINDOW // 2
ax[1].plot(ctr, S["knn"]["bad"], color="C3"); shade(ax[1], starts, S["knn"]["pb"]); ax[1].set_title("k-NN novelty score on corrupted data (red = flagged)", loc="left")
ax[2].plot(ctr, S["knn"]["clean"], color="C0"); shade(ax[2], starts, S["knn"]["pc"]); ax[2].set_title("k-NN novelty score on the ORIGINAL data (red = flagged)", loc="left")
for a in ax: a.grid(alpha=.3)
ax[2].set_xlabel("sample"); plt.tight_layout(); plt.show()

### Sensitivity: how does the threshold multiplier trade precision against recall?
Changing the multiplier only moves the alarm line, so we can sweep it using the scores above (thresholds taken from a model trained on all windows).

In [ ]:
knn_all = KNNNovelty().fit(F)
sweep = []
for m in np.round(np.arange(0.5, 5.01, 0.25), 2):
    p = S["knn"]["bad"] > knn_all.q99 * m
    sweep.append((m, precision_score(y_true, p, zero_division=0), recall_score(y_true, p, zero_division=0), f1_score(y_true, p, zero_division=0),
                  (S["knn"]["clean"] > knn_all.q99 * m).mean()))
sw = pd.DataFrame(sweep, columns=["multiplier", "precision", "recall", "F1", "flagged in original"])
plt.figure(figsize=(8, 3.5))
for c in ["precision", "recall", "F1", "flagged in original"]: plt.plot(sw.multiplier, sw[c], marker="o", ms=3, label=c)
plt.axvline(THRESH_MULT, color="k", ls=":"); plt.xlabel("threshold multiplier"); plt.legend(fontsize=8); plt.grid(alpha=.3); plt.show()
sw.loc[sw.F1.idxmax()]

## 7 · Which sensors caused the alarms? (explainability)
For every window flagged on the corrupted data, we compare it with its nearest normal window and see which channel differs most.

In [ ]:
flag_bad = S["knn"]["pb"]
if flag_bad.any():
    contrib = knn_all.explain(F_bad[flag_bad]).mean(0)
    order = np.argsort(contrib)[::-1]
    plt.figure(figsize=(8, 3.2)); plt.bar(np.array(CHANNELS)[order], contrib[order], color="C0")
    plt.title("Mean deviation from normal in flagged windows (std. units)"); plt.xticks(rotation=30); plt.tight_layout(); plt.show()
    print("Top 3 channels:", [CHANNELS[i] for i in order[:3]])
else:
    print("No flagged windows.")

## 8 · Save the model and the results

In [ ]:
import joblib
final_model = KNNNovelty().fit(F)                       # final model: trained on the whole recording
joblib.dump(final_model, "imu_knn_novelty_model.joblib")
out = pd.DataFrame({"start_sample": starts, "knn_score_blocked_cv": S["knn"]["clean"], "knn_flag": S["knn"]["pc"],
                    "iso_score_blocked_cv": S["iso"]["clean"], "iso_flag": S["iso"]["pc"]})
out.to_csv("anomaly_results.csv", index=False)
results.to_csv("evaluation_summary.csv")
print("saved: imu_knn_novelty_model.joblib, anomaly_results.csv, evaluation_summary.csv")

## What to say when asked "how do you know the model is good?"
> The dataset is unlabeled, so we cannot report a supervised accuracy. Instead we (1) score every window with **blocked cross-validation** so no window is judged by a model that saw it,
> (2) **inject known faults** and measure **precision, recall, F1 and ROC-AUC**, (3) report how much of the untouched recording gets flagged, and (4) compare two independent detectors
> (k-NN novelty and Isolation Forest). The sensitivity sweep shows the trade-off, and the channel plot explains *why* a window was flagged.

**Limitations to state honestly:** the injected faults are synthetic (and fairly obvious); we assume the 1000-row recording is mostly normal; a movement that occurs only once
in the file is harder to judge because few similar windows remain after the gap. Results should be repeated with separate normal / test recordings from the Kaggle dataset.